# Tetris ASCII
Execute a célula, clique no desenho. O jogo começa automaticamente. Use as setas ou botões.

Python exibe HTML; JavaScript controla o jogo no navegador. Não precisa instalar pacotes.

A cobrinha atravessa as bordas e o próprio corpo. O Tetris usa blocos ASCII, o Pong é contra o computador e o RPG pede uma chave para abrir a porta.

In [ ]:
from IPython.display import Javascript, display

# Execute esta célula inteira. Clique no desenho e use as setas.
# Python exibe a saída; JavaScript controla o jogo no navegador.
display(Javascript(r"""(() => {
// O painel fica na saída desta célula, sem iframe dentro de iframe.
const painel = document.createElement('div');
painel.style = 'background:#111;color:#a6ff99;padding:12px;max-width:100%;font:14px monospace';
painel.innerHTML = "<b>Tetris ASCII</b><p><button data-action=\"restart\">Reiniciar</button> <button data-action=\"pause\">Pausar / continuar</button></p><span role=\"status\"></span><pre tabindex=\"0\" aria-label=\"Jogo ASCII: use as setas\" style=\"font:14px/1.2 monospace;overflow:auto;outline:1px solid #486;padding:6px\"></pre><div><button data-key=\"ArrowUp\">Cima</button><button data-key=\"ArrowDown\">Baixo</button><button data-key=\"ArrowLeft\">Esquerda</button><button data-key=\"ArrowRight\">Direita</button><button data-key=\" \">Soltar</button></div>";
document.body.appendChild(painel);
const tela = painel.querySelector('pre'), aviso = painel.querySelector('[role="status"]');
let tempo = null, rodando = false, terminou = false;
function moldura(linhas) {
    const borda = '+' + '-'.repeat(linhas[0].length) + '+';
    return [borda, ...linhas.map(l => '|' + l + '|'), borda].join(String.fromCharCode(10));
}
// O tabuleiro é uma matriz. # é uma peça fixa e @ é a peça caindo.
const largura = 10, altura = 16;
const formas = [ [[1,1,1,1]], [[1,1],[1,1]], [[0,1,0],[1,1,1]], [[1,0,0],[1,1,1]], [[0,0,1],[1,1,1]], [[0,1,1],[1,1,0]], [[1,1,0],[0,1,1]] ];
let tabuleiro, peca, x, y, pontos;
function cabe(px, py, forma) {
    return forma.every((linha, dy) => linha.every((bloco, dx) => !bloco ||
        (px+dx >= 0 && px+dx < largura && py+dy >= 0 && py+dy < altura && !tabuleiro[py+dy][px+dx])));
}
function novaPeca() {
    peca = formas[Math.floor(Math.random()*formas.length)].map(linha => [...linha]);
    x = Math.floor((largura - peca[0].length) / 2); y = 0;
    if (!cabe(x,y,peca)) terminou = true;
}
function reiniciar() {
    tabuleiro = Array.from({length:altura}, () => Array(largura).fill(0));
    pontos = 0; novaPeca();
}
function fixar() {
    peca.forEach((linha,dy) => linha.forEach((bloco,dx) => { if (bloco) tabuleiro[y+dy][x+dx] = 1; }));
    const restantes = tabuleiro.filter(linha => linha.some(bloco => bloco === 0));
    pontos += (altura - restantes.length) * 100;
    while (restantes.length < altura) restantes.unshift(Array(largura).fill(0));
    tabuleiro = restantes; novaPeca();
}
function atualizar() {
    if (cabe(x,y+1,peca)) y++;
    else fixar();
}
function comando(tecla) {
    if (tecla === 'ArrowLeft' && cabe(x-1,y,peca)) x--;
    if (tecla === 'ArrowRight' && cabe(x+1,y,peca)) x++;
    if (tecla === 'ArrowDown') atualizar();
    if (tecla === 'ArrowUp') {
        const girada = peca[0].map((_,coluna) => peca.map(linha => linha[coluna]).reverse());
        if (cabe(x,y,girada)) peca = girada;
    }
    if (tecla === ' ') {
        while (cabe(x,y+1,peca)) y++;
        fixar();
    }
}
function desenhar() {
    const desenho = tabuleiro.map(linha => linha.map(bloco => bloco ? '#' : '.'));
    if (!terminou) peca.forEach((linha,dy) => linha.forEach((bloco,dx) => { if (bloco) desenho[y+dy][x+dx] = '@'; }));
    return moldura(desenho.map(linha => linha.join(''))) + '\nPontos: ' + pontos + '\n' + (terminou ? 'FIM DE JOGO: o tabuleiro encheu.' : 'Cima: girar | Baixo: descer\nEsquerda/direita: mover | Espaco: soltar');
}

function parar() { clearInterval(tempo); tempo = null; rodando = false; }
function mostrar() {
    tela.textContent = desenhar();
    if (terminou) { parar(); aviso.textContent = 'Fim da partida. Use Reiniciar.'; }
}
function continuar() {
    if (terminou) return;
    parar(); rodando = true;
    aviso.textContent = 'Clique no desenho e use as setas, ou toque nos botoes.';
    tempo = setInterval(() => { if (!painel.isConnected) { parar(); return; } atualizar(); mostrar(); }, 500);
    tela.focus();
}
function iniciar() { parar(); terminou = false; reiniciar(); mostrar(); continuar(); }
function receber(tecla) { if (rodando && !terminou) { comando(tecla); mostrar(); } }
tela.onkeydown = e => {
    if (['ArrowUp','ArrowDown','ArrowLeft','ArrowRight',' '].includes(e.key)) {
        e.preventDefault(); receber(e.key);
    }
};
painel.querySelectorAll('[data-key]').forEach(b => b.onclick = () => { receber(b.dataset.key); tela.focus(); });
painel.querySelector('[data-action="restart"]').onclick = iniciar;
painel.querySelector('[data-action="pause"]').onclick = () => {
    if (rodando) { parar(); aviso.textContent = 'Pausado.'; } else continuar();
};
window.addEventListener('pagehide', parar, {once:true});
iniciar();
})();"""))